# EDA — Default of Credit Card Clients

**Source:** UCI *Default of Credit Card Clients* — 30,000 credit card holders in Taiwan, covering
the billing months April to September 2005.

**One row = one person.** The question the data answers: did they miss their payment the *following*
month, October 2005?

**This notebook only looks. It never changes anything.** It loads the file exactly as it ships and
never renames, recodes, drops or saves anything. All the changes live in `preprocessing.ipynb`.
Keeping them apart means you can always answer "what is actually in this file?" without wondering
which edits have already been applied.

**Why the first attempt to load it failed:** `data/credit_card_clients.xls` is a genuine old-style
Excel workbook, not a CSV with an `.xls` name stuck on the end. `pd.read_csv` tried to read it as
text and choked on the very first byte. It needs `pd.read_excel`, which for `.xls` files relies on
the `xlrd` package (now added to `pyproject.toml`).

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

## 1. Load the data

Two quirks of this file:

- it is an Excel workbook, so use `read_excel` (not `read_csv`), and read the sheet named `Data`
- row 0 holds the placeholder names (`X1`, `X2`, …) and the real column names are on row 1, hence
  `header=1`

The target column arrives as `default payment next month`, spaces and all. This notebook keeps that
name as-is and refers to it through the `TARGET` variable. Renaming it counts as a change, so that
happens in `preprocessing.ipynb`.

In [2]:
DATA_PATH = Path("../data/credit_card_clients.xls")
TARGET = "default payment next month"

dataset = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)

dataset.shape

(30000, 25)

In [3]:
dataset.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. What the columns mean

The official documentation calls the input variables `X1` to `X23`. Those names are literally row 0
of the spreadsheet, so the table below is read straight off the file rather than guessed:

| docs | column | meaning |
| --- | --- | --- |
| — | `ID` | row number — not a real feature, drop it before training |
| X1 | `LIMIT_BAL` | credit limit in NT dollars (their card plus any family cards) |
| X2 | `SEX` | 1 = male, 2 = female |
| X3 | `EDUCATION` | 1 = graduate school, 2 = university, 3 = high school, 4 = others |
| X4 | `MARRIAGE` | 1 = married, 2 = single, 3 = others |
| X5 | `AGE` | age in years |
| X6–X11 | `PAY_0`, `PAY_2` … `PAY_6` | how far behind on payments, for Sep, Aug, Jul, Jun, May, Apr |
| X12–X17 | `BILL_AMT1` … `BILL_AMT6` | how much they owed on that month's statement, Sep … Apr |
| X18–X23 | `PAY_AMT1` … `PAY_AMT6` | how much they actually paid that month, Sep … Apr |
| Y | `default payment next month` | **the answer**: 1 = missed the October payment, 0 = paid it |

That is **23 input variables + 1 answer + `ID` = 25 columns.**

The three month-by-month groups are the same months seen three ways: **what they owed**
(`BILL_AMT`), **what they paid** (`PAY_AMT`), and **how far behind that left them** (`PAY_`).

Two things that catch people out:

- **The month columns run backwards.** `PAY_0`, `BILL_AMT1` and `PAY_AMT1` are all the most recent
  month (September). Number 6 is the oldest (April). And there is no `PAY_1` — September's column
  is called `PAY_0`. That odd naming comes from the original dataset, it is not a mistake here.
- **The `PAY_*` codes** are documented as `-1` = paid on time, `1` = one month late, up to `9` =
  nine months or more. Section 3 shows the file does not actually stick to that.

## 3. Does the file match the documentation?

Worth checking before trusting any of it. The documentation gives a fixed list of codes for `SEX`,
`EDUCATION`, `MARRIAGE` and the `PAY_*` columns. The cell below compares that list against what
actually turns up in the 30,000 rows, both ways round: codes that appear but are not documented,
and documented codes that never appear.

In [4]:
PAY_COLUMNS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# The codes as published, before looking at the file.
CODEBOOK = {
    "SEX": {1, 2},
    "EDUCATION": {1, 2, 3, 4},
    "MARRIAGE": {1, 2, 3},
    **{column: {-1, *range(1, 10)} for column in PAY_COLUMNS},
}

audit = [] 
for column, documented in CODEBOOK.items():
    observed = set(dataset[column].unique())
    undocumented = sorted(observed - documented)
    audit.append(
        {
            "column": column,
            "undocumented_codes": undocumented or "-",
            "rows_affected": int(dataset[column].isin(undocumented).sum()),
            "share_of_rows": round(dataset[column].isin(undocumented).mean(), 3),
            "documented_but_absent": sorted(documented - observed) or "-",
        }
    )

pd.DataFrame(audit).set_index("column")

,undocumented_codes,rows_affected,share_of_rows,documented_but_absent
column,,,,
SEX,-,0,0.000,-
EDUCATION,"[0, 5, 6]",345,0.012,-
MARRIAGE,[0],54,0.002,-
PAY_0,"[-2, 0]",17496,0.583,[9]
PAY_2,"[-2, 0]",19512,0.650,[9]
PAY_3,"[-2, 0]",19849,0.662,[9]
PAY_4,"[-2, 0]",20803,0.693,[9]
PAY_5,"[-2, 0]",21493,0.716,"[1, 9]"
PAY_6,"[-2, 0]",21181,0.706,"[1, 9]"


The documentation is correct as far as it goes, but it leaves things out:

- **`SEX`** is the only clean one — exactly `1` and `2`, nothing else.
- **`EDUCATION`** also contains `0`, `5` and `6` (345 people). None of those are explained.
- **`MARRIAGE`** also contains `0` (54 people). Also unexplained.
- **`PAY_*`** is the big one. The documented codes are `-1` and `1` to `9`, but the file also uses
  `-2` and `0`, and those two make up **58% to 72% of every month** — `0` on its own is the most
  common value in all six columns. Meanwhile the documented code `9` never appears at all.

Section 8 digs into what `-2` and `0` actually mean, since between them they cover most of the data.

In [5]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

## 4. Data quality

Checking the range of each column matters more than counting missing values here. There are no
missing values at all, but there are the odd category codes section 3 just found, plus some negative
bill amounts (which turn out to be overpayments and refunds).

In [6]:
quality = pd.DataFrame({
    "dtype": dataset.dtypes.astype(str),
    "missing": dataset.isna().sum(),
    "unique": dataset.nunique(),
    "min": dataset.min(numeric_only=True),
    "max": dataset.max(numeric_only=True),
})
quality

,dtype,missing,unique,min,max
ID,int64,0,30000,1,30000
LIMIT_BAL,int64,0,81,10000,1000000
SEX,int64,0,2,1,2
EDUCATION,int64,0,7,0,6
MARRIAGE,int64,0,4,0,3
AGE,int64,0,56,21,79
PAY_0,int64,0,11,-2,8
PAY_2,int64,0,11,-2,8
PAY_3,int64,0,11,-2,8
PAY_4,int64,0,11,-2,8


In [7]:
print("rows, columns          :", dataset.shape)
print("missing cells          :", int(dataset.isna().sum().sum()))
print("duplicate rows         :", int(dataset.duplicated().sum()))
print("duplicate IDs          :", int(dataset["ID"].duplicated().sum()))
print("duplicates ignoring ID :", int(dataset.drop(columns=["ID"]).duplicated().sum()))

rows, columns          : (30000, 25)
missing cells          : 0
duplicate rows         : 0
duplicate IDs          : 0
duplicates ignoring ID : 35


## 5. How many people actually default?

This is the most important number for modelling, because the two answers are not evenly split —
roughly one in five defaults.

That matters more than it sounds. A model that simply guessed "nobody ever defaults" would be right
78% of the time, which looks like a good score and is completely useless. So accuracy is the wrong
thing to measure here. Use ROC-AUC and precision/recall instead.

In [8]:
counts = dataset[TARGET].value_counts().sort_index()
share = dataset[TARGET].value_counts(normalize=True).sort_index()

pd.DataFrame({"clients": counts, "share": share.round(4)}).rename(
    index={0: "0 - paid", 1: "1 - defaulted"}
)

,clients,share
default payment next month,,
0 - paid,23364,0.7788
1 - defaulted,6636,0.2212


## 6. Who is in the data?

The categories are stored as bare numbers, which are hard to read, so we swap in labels first.
`labelled` is a throwaway copy used only for this section — `dataset` itself is never modified.

In [9]:
SEX_LABELS = {1: "male", 2: "female"}
EDUCATION_LABELS = {
    1: "graduate school", 2: "university", 3: "high school", 4: "other",
    0: "undocumented", 5: "undocumented", 6: "undocumented",
}
MARRIAGE_LABELS = {1: "married", 2: "single", 3: "other", 0: "undocumented"}

labelled = dataset.assign(
    SEX_LABEL=dataset["SEX"].map(SEX_LABELS),
    EDUCATION_LABEL=dataset["EDUCATION"].map(EDUCATION_LABELS),
    MARRIAGE_LABEL=dataset["MARRIAGE"].map(MARRIAGE_LABELS),
)

for column in ["SEX_LABEL", "EDUCATION_LABEL", "MARRIAGE_LABEL"]:
    print(labelled[column].value_counts(dropna=False), end="\n\n")

SEX_LABEL
female    18112
male      11888
Name: count, dtype: int64

EDUCATION_LABEL
university         14030
graduate school    10585
high school         4917
undocumented         345
other                123
Name: count, dtype: int64

MARRIAGE_LABEL
single          15964
married         13659
other             323
undocumented       54
Name: count, dtype: int64



## 7. Default rate by group

The counts on their own do not tell you much. What matters is how the default rate shifts between
groups.

Always read the `clients` column alongside it: a 26% default rate based on 323 people is far weaker
evidence than the same 26% based on 14,000 people.

In [10]:
def default_rate_by(column, frame=labelled):
    summary = frame.groupby(column, observed=True)[TARGET].agg(
        clients="size", default_rate="mean"
    )
    return summary.assign(default_rate=summary["default_rate"].round(3)).sort_values(
        "clients", ascending=False
    )


for column in ["SEX_LABEL", "EDUCATION_LABEL", "MARRIAGE_LABEL"]:
    print(default_rate_by(column), end="\n\n")

           clients  default_rate
SEX_LABEL                       
female       18112         0.208
male         11888         0.242

                 clients  default_rate
EDUCATION_LABEL                       
university         14030         0.237
graduate school    10585         0.192
high school         4917         0.252
undocumented         345         0.075
other                123         0.057

                clients  default_rate
MARRIAGE_LABEL                       
single            15964         0.209
married           13659         0.235
other               323         0.260
undocumented         54         0.093



In [11]:
age_bands = pd.cut(dataset["AGE"], bins=[20, 30, 40, 50, 60, 100], right=False)

dataset.groupby(age_bands, observed=True)[TARGET].agg(
    clients="size", default_rate="mean"
).round(3)

,clients,default_rate
AGE,,
"[20, 30)",9618,0.228
"[30, 40)",11238,0.203
"[40, 50)",6464,0.230
"[50, 60)",2341,0.249
"[60, 100)",339,0.283


## 8. Payment history — the strongest signal

The `PAY_*` columns are where most of the predictive power sits. Most people sit in `-2`, `-1` or
`0`, and the "months late" codes drop off quickly after that.

Since `-2` and `0` are undocumented but cover most of the data, the second cell below works out what
they mean from the numbers themselves. It compares September's payment against August's bill for
each code.

In [12]:
pay_status = pd.DataFrame(
    {column: dataset[column].value_counts() for column in PAY_COLUMNS}
).sort_index().fillna(0).astype(int)
pay_status

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
-2,2759,3782,4085,4348,4546,4895
-1,5686,6050,5938,5687,5539,5740
0,14737,15730,15764,16455,16947,16286
1,3688,28,4,2,0,0
2,2667,3927,3819,3159,2626,2766
3,322,326,240,180,178,184
4,76,99,76,69,84,49
5,26,25,21,35,17,13
6,11,12,23,5,4,19
7,9,20,27,58,58,46


In [13]:
# PAY_0 is last month's repayment status - the most recent, most informative column.
dataset.groupby("PAY_0")[TARGET].agg(clients="size", default_rate="mean").round(3)

,clients,default_rate
PAY_0,,
-2,2759,0.132
-1,5686,0.168
0,14737,0.128
1,3688,0.339
2,2667,0.691
3,322,0.758
4,76,0.684
5,26,0.500
6,11,0.545


In [14]:
# September's payment (PAY_AMT1) settles August's bill (BILL_AMT2), so comparing the two
# shows what each status code actually describes.
recent = dataset[dataset["PAY_0"].between(-2, 2)]

recent.groupby("PAY_0").agg(
    clients=("PAY_0", "size"),
    august_bill=("BILL_AMT2", "median"),
    september_payment=("PAY_AMT1", "median"),
    paid_nothing=("PAY_AMT1", lambda s: (s == 0).mean().round(2)),
).assign(
    paid_in_full=recent.assign(full=recent["PAY_AMT1"] >= recent["BILL_AMT2"])
    .groupby("PAY_0")["full"]
    .mean()
    .round(2)
)

,clients,august_bill,september_payment,paid_nothing,paid_in_full
PAY_0,,,,,
-2,2759,1099.0,1131.0,0.33,0.95
-1,5686,2373.5,1732.5,0.18,0.80
0,14737,48261.0,3000.0,0.02,0.05
1,3688,6351.5,0.0,0.59,0.50
2,2667,41387.0,2000.0,0.20,0.07


That table explains the two undocumented codes:

- **`-2`** — tiny bill, paid in full. The card was barely used that month.
- **`-1`** — a real bill, paid off in full. This is "paid on time" as documented.
- **`0`** — owed around NT$48,000 and paid around NT$3,000. That is roughly 6%, i.e. the minimum
  payment. Only 2% of them paid nothing at all, so they *are* paying — just not clearing the debt.
  This is revolving credit.

So `-2`, `-1` and `0` are not a scale from bad to good. They are three different situations. Only
`1` and above is an actual count of months behind.

The surprise: code `0` has the **lowest** default rate of the three (12.8%, versus 16.8% for people
who pay in full). People who carry a balance and keep paying it down are the bank's most reliable
customers.

Note that this reading is worked out from the numbers, not from the documentation — so treat it as
a well-supported assumption rather than a fact.

## 9. The money columns

Bills and payments are very lopsided: the typical bill is around NT$22,000, but the largest is over
NT$1.6 million. A handful of people are far above everyone else.

Negative `BILL_AMT` values are overpayments and refunds, not errors — `PAY_AMT` has none.

This lopsidedness is worth remembering when picking a model. Tree-based models cope with it fine;
linear models usually want the numbers scaled or log-transformed first.

In [15]:
MONEY_COLUMNS = (
    ["LIMIT_BAL"]
    + [f"BILL_AMT{i}" for i in range(1, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 7)]
)

dataset[["AGE"] + MONEY_COLUMNS].describe().T.round(0)

,count,mean,std,min,25%,50%,75%,max
AGE,30000.0,35.0,9.0,21.0,28.0,34.0,41.0,79.0
LIMIT_BAL,30000.0,167484.0,129748.0,10000.0,50000.0,140000.0,240000.0,1000000.0
BILL_AMT1,30000.0,51223.0,73636.0,-165580.0,3559.0,22382.0,67091.0,964511.0
BILL_AMT2,30000.0,49179.0,71174.0,-69777.0,2985.0,21200.0,64006.0,983931.0
BILL_AMT3,30000.0,47013.0,69349.0,-157264.0,2666.0,20088.0,60165.0,1664089.0
BILL_AMT4,30000.0,43263.0,64333.0,-170000.0,2327.0,19052.0,54506.0,891586.0
BILL_AMT5,30000.0,40311.0,60797.0,-81334.0,1763.0,18104.0,50190.0,927171.0
BILL_AMT6,30000.0,38872.0,59554.0,-339603.0,1256.0,17071.0,49198.0,961664.0
PAY_AMT1,30000.0,5664.0,16563.0,0.0,1000.0,2100.0,5006.0,873552.0
PAY_AMT2,30000.0,5921.0,23041.0,0.0,833.0,2009.0,5000.0,1684259.0


## 10. What is linked to defaulting?

Correlation only picks up straight-line relationships, so treat this as a rough ranking rather than
proof of anything. It still makes the overall picture clear.

Read it as: positive = higher value goes with more defaults, negative = higher value goes with
fewer.

In [16]:
correlations = (
    dataset.drop(columns=["ID"])
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
)

correlations.sort_values(key=abs, ascending=False).round(3).to_frame("corr_with_default")

,corr_with_default
PAY_0,0.325
PAY_2,0.264
PAY_3,0.235
PAY_4,0.217
PAY_5,0.204
PAY_6,0.187
LIMIT_BAL,-0.154
PAY_AMT1,-0.073
PAY_AMT2,-0.059
PAY_AMT4,-0.057


## 11. Takeaways

1. **30,000 rows, 23 input columns plus the answer, and no missing values.** No rows are completely
   identical; 35 rows match once you ignore `ID`, which is normal at this size. The data is clean,
   so the work is in how we frame it, not in repairing it.
2. **The documentation is incomplete** (section 3). `EDUCATION` has 345 people in codes `0`/`5`/`6`,
   `MARRIAGE` has 54 in code `0`, and every `PAY_*` column uses `-2` and `0`, with `0` being the most
   common value in all six months. Documented code `9` never appears; the worst seen is `8`.
   What to do about that is a decision for `preprocessing.ipynb`.
3. **Only 22% of people default.** Because of that, accuracy is misleading — use ROC-AUC and
   precision/recall, and consider `class_weight="balanced"` when training.
4. **Recent payment history matters most.** `PAY_0` (September) is the strongest single column, and
   the default rate climbs from around 13% for people who are up to date to 69% for people two
   months behind. The older months matter progressively less (0.33 for September down to 0.19 for
   April).
5. **A higher credit limit means fewer defaults** (-0.15). That mostly reflects the bank's own
   judgement — it gave bigger limits to people it already trusted.
6. **Age, sex and education barely matter** — only a few percentage points between groups, and the
   most extreme rates come from tiny groups (`EDUCATION` code 0 is just 14 people). Do not read much
   into them.
7. **The raw bill and payment amounts barely matter on their own.** Every correlation is under 0.08.
   Combining them probably will matter though: payment divided by bill, bill divided by credit
   limit, or how those change across the six months.

**Next:** `preprocessing.ipynb` applies the changes this notebook only suggests, then step 1 of
`STEPS.md` — training a model and logging the runs to MLflow.